# Two-Layer Neural Network with Keras (MNIST)
Implement `TwoLayerNetKeras`, train it on the MNIST training set, and test it on the MNIST test set.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Load the MNIST dataset

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)

In [ ]:
plt.figure(figsize=(8, 2))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.show()

## 2. Preprocess

In [ ]:
# flatten 28x28 -> 784 and scale pixels to [0, 1]
X_train = X_train.reshape(-1, 28 * 28).astype("float32") / 255
X_test = X_test.reshape(-1, 28 * 28).astype("float32") / 255
print("X_train:", X_train.shape, "X_test:", X_test.shape)

## 3. `TwoLayerNetKeras` class
Two layers = one hidden Dense layer (ReLU) + one output Dense layer (softmax).

In [ ]:
class TwoLayerNetKeras:
    def __init__(self, input_size=784, hidden_size=100, output_size=10):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.model = None
        self.history = None

    def create_nn(self):
        self.model = keras.Sequential([
            keras.Input(shape=(self.input_size,)),
            layers.Dense(self.hidden_size, activation="relu"),  # layer 1: hidden
            layers.Dense(self.output_size, activation="softmax"),  # layer 2: output
        ])
        self.model.compile(optimizer="rmsprop",
                           loss="sparse_categorical_crossentropy",  # labels are integers 0-9
                           metrics=["accuracy"])
        self.model.summary()

    def train(self, X, y, epochs=10, batch_size=64, validation_split=0.2):
        self.history = self.model.fit(X, y, epochs=epochs, batch_size=batch_size,
                                      validation_split=validation_split, verbose=2)
        return self.history

    def evaluate(self, X, y):
        return self.model.evaluate(X, y, verbose=0)

    def predict(self, X):
        return np.argmax(self.model.predict(X, verbose=0), axis=1)  # class with highest probability

In [ ]:
net = TwoLayerNetKeras(input_size=784, hidden_size=100, output_size=10)
net.create_nn()

## 4. Train on the MNIST training set

In [ ]:
history = net.train(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], marker="o", label="Training loss")
plt.plot(history.history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss"); plt.legend(); plt.grid()
plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], marker="o", label="Training acc")
plt.plot(history.history["val_accuracy"], marker="o", label="Validation acc")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Accuracy"); plt.legend(); plt.grid()
plt.show()

## 5. Test on the MNIST test set

In [ ]:
test_loss, test_acc = net.evaluate(X_test, y_test)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy (evaluate): {test_acc:.4f}")

# double-check by comparing predicted classes with true labels
pred_class = net.predict(X_test)
print(f"Test accuracy (manual):   {np.mean(pred_class == y_test):.4f}")
print(f"Correct: {np.sum(pred_class == y_test)} / {len(y_test)}")

## 6. Prediction examples

In [ ]:
plt.figure(figsize=(8, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap="gray")
    color = "green" if pred_class[i] == y_test[i] else "red"
    plt.title(f"True: {y_test[i]}, Pred: {pred_class[i]}", color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# a few misclassified test images
wrong = np.where(pred_class != y_test)[0][:6]
plt.figure(figsize=(12, 2.5))
for j, i in enumerate(wrong):
    plt.subplot(1, 6, j + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap="gray")
    plt.title(f"True: {y_test[i]}, Pred: {pred_class[i]}", color="red")
    plt.axis("off")
plt.show()

## 7. Conclusion
- The loss goes down and the accuracy goes up every epoch, so training works.
- The test accuracy above (on 10,000 unseen images) is close to the validation accuracy, so testing works and the model generalizes.